In [1]:
import joblib
import numpy as np
import pandas as pd
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')



In [2]:
# ============================================================================
# STEP 1: LOAD MODEL AND DATA
# ============================================================================
print("=" * 60)
print("STEP 1: LOADING MODEL AND DATA")
print("=" * 60)

try:
    model = keras.models.load_model("../models/best_ann_model.keras")
    print("✅ Model loaded successfully")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("   Please run 02_ann_model.ipynb first")

try:
    X_test = joblib.load("../models/X_test_processed.pkl")
    y_test = joblib.load("../models/y_test_processed.pkl")
    optimal_threshold = joblib.load("../models/optimal_threshold.pkl")
    print("✅ Test data loaded successfully")
except Exception as e:
    print(f"❌ Error loading test data: {e}")

# Load original data for customer information
try:
    df_original = pd.read_csv("../data/processed/processed_churn_data.csv")
    print(f"✅ Original data loaded: {df_original.shape}")
except Exception as e:
    print(f"❌ Error loading original data: {e}")
    # Create synthetic data if file doesn't exist
    print("   Creating synthetic data...")
    df_original = pd.DataFrame({
        'Age': np.random.randint(18, 70, 2000),
        'Balance': np.random.uniform(0, 200000, 2000),
        'NumOfProducts': np.random.randint(1, 5, 2000),
        'IsActiveMember': np.random.choice([0, 1], 2000),
        'CreditScore': np.random.randint(350, 850, 2000),
        'Geography': np.random.choice([0, 1, 2], 2000),
        'Gender': np.random.choice([0, 1], 2000)
    })


STEP 1: LOADING MODEL AND DATA
✅ Model loaded successfully
✅ Test data loaded successfully
✅ Original data loaded: (10000, 18)


In [3]:
# ============================================================================
# STEP 2: PREPARE CUSTOMER DATA FOR PREDICTIONS
# ============================================================================
print("\n" + "=" * 60)
print("STEP 2: PREPARING CUSTOMER DATA")
print("=" * 60)

# Get the test customers (last 20% of the dataset)
test_size = len(y_test)
X_test_original = df_original.iloc[-test_size:].reset_index(drop=True)
print(f"✅ Test customers: {len(X_test_original)}")

# Convert to numpy arrays for prediction
if isinstance(X_test, pd.DataFrame):
    X_test = X_test.values
if isinstance(y_test, pd.Series):
    y_test = y_test.values



STEP 2: PREPARING CUSTOMER DATA
✅ Test customers: 2000


In [4]:
# ============================================================================
# STEP 3: GENERATE PREDICTIONS
# ============================================================================
print("\n" + "=" * 60)
print("STEP 3: GENERATING PREDICTIONS")
print("=" * 60)

y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > optimal_threshold).astype(int)

print(f"✅ Predictions generated for {len(y_pred_prob)} customers")


STEP 3: GENERATING PREDICTIONS
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
✅ Predictions generated for 2000 customers


In [5]:
# ============================================================================
# STEP 4: DEFINE RISK LEVEL FUNCTION
# ============================================================================
print("\n" + "=" * 60)
print("STEP 4: DEFINING RISK LEVELS")
print("=" * 60)

def get_risk_level(probability):
    """Categorize customers by risk level based on churn probability"""
    if probability < 0.3:
        return "Low Risk"
    elif probability < 0.7:
        return "Medium Risk"
    else:
        return "High Risk"

print("✅ Risk levels defined:")
print("   - Low Risk: < 30%")
print("   - Medium Risk: 30-70%")
print("   - High Risk: > 70%")



STEP 4: DEFINING RISK LEVELS
✅ Risk levels defined:
   - Low Risk: < 30%
   - Medium Risk: 30-70%
   - High Risk: > 70%


In [6]:
# ============================================================================
# STEP 5: CREATE CUSTOMER PROFILES
# ============================================================================
print("\n" + "=" * 60)
print("STEP 5: CREATING CUSTOMER PROFILES")
print("=" * 60)

customer_profiles = []
num_customers = min(50, len(X_test))  # Generate for 50 customers

for i in range(num_customers):
    profile = {
        'customer_id': f"CUST_{1001 + i}",
        'age': X_test_original.iloc[i]['Age'] if 'Age' in X_test_original.columns else 45,
        'balance': X_test_original.iloc[i]['Balance'] if 'Balance' in X_test_original.columns else 0,
        'products': X_test_original.iloc[i]['NumOfProducts'] if 'NumOfProducts' in X_test_original.columns else 1,
        'active': bool(X_test_original.iloc[i]['IsActiveMember']) if 'IsActiveMember' in X_test_original.columns else False,
        'credit_score': X_test_original.iloc[i]['CreditScore'] if 'CreditScore' in X_test_original.columns else 600,
        'geography': X_test_original.iloc[i]['Geography'] if 'Geography' in X_test_original.columns else 0,
        'gender': X_test_original.iloc[i]['Gender'] if 'Gender' in X_test_original.columns else 0,
        'churn_probability': y_pred_prob[i],
        'predicted_churn': 'Yes' if y_pred[i] == 1 else 'No',
        'actual_churn': 'Yes' if y_test[i] == 1 else 'No',
        'risk_level': get_risk_level(y_pred_prob[i])
    }
    customer_profiles.append(profile)

print(f"✅ Created profiles for {len(customer_profiles)} customers")


STEP 5: CREATING CUSTOMER PROFILES
✅ Created profiles for 50 customers


In [7]:
# ============================================================================
# STEP 6: TEMPLATE-BASED RETENTION STRATEGIES (100% RELIABLE)
# ============================================================================
print("\n" + "=" * 60)
print("STEP 6: GENERATING TEMPLATE-BASED STRATEGIES")
print("=" * 60)

def get_retention_strategy(customer):
    """Generate personalized retention strategy using templates (ALWAYS WORKS)"""
    
    # Extract customer data
    risk = customer['risk_level']
    products = customer['products']
    balance = customer['balance']
    active = customer['active']
    age = customer['age']
    credit = customer['credit_score']
    prob = customer['churn_probability']
    
    # Geography mapping
    geo_map = {0: "France", 1: "Germany", 2: "Spain"}
    geography = geo_map.get(customer['geography'], "Unknown")
    
    # Gender mapping
    gender = "Male" if customer['gender'] == 0 else "Female"
    
    # Strategy header
    strategy = f"""
{'='*60}
🎯 RETENTION STRATEGY FOR {customer['customer_id']}
{'='*60}

📊 CUSTOMER SUMMARY:
────────────────────────────────────────────────
• Risk Level: {risk}
• Churn Probability: {prob:.1%}
• Age: {age} | Gender: {gender} | Location: {geography}
• Products: {products} | Active Member: {'Yes' if active else 'No'}
• Balance: ${balance:,.2f} | Credit Score: {credit}

"""
    
    # HIGH RISK CUSTOMERS (Urgent Intervention)
    if risk == "High Risk":
        strategy += """
🔴 URGENT INTERVENTION REQUIRED
────────────────────────────────────────────────

📋 RISK ANALYSIS:
• This customer is at HIGH RISK of churn with {prob:.1%} probability
• {reason}
• Immediate action required within 24-48 hours

🎯 RECOMMENDED RETENTION ACTIONS:
"""
        # Add personalized reasons
        reasons = []
        if not active:
            reasons.append("Customer is INACTIVE - needs re-engagement")
        if products <= 1:
            reasons.append("Low product adoption (only 1 product)")
        if balance > 50000:
            reasons.append("High-value customer with significant balance at risk")
        if age > 60:
            reasons.append("Senior customer - may need different engagement approach")
        
        reason_text = reasons[0] if reasons else "Multiple risk factors detected"
        strategy = strategy.replace("{reason}", reason_text)
        
        strategy += """
1️⃣ IMMEDIATE OUTREACH:
   • Priority phone call within 24 hours
   • Personal banker assigned for follow-up
   • Welcome package with personalized offers

2️⃣ PREMIUM RETENTION OFFER:
"""
        if balance > 50000:
            strategy += """   • Upgrade to Premium Banking - first 6 months FREE
   • Dedicated relationship manager
   • Exclusive investment opportunities with reduced fees
"""
        else:
            strategy += """   • Cashback offer: $100 on next 10 transactions
   • Waive monthly fees for 12 months
   • 0% interest on balance transfers for 6 months
"""
        
        if products <= 1:
            strategy += """
3️⃣ PRODUCT BUNDLE OFFER:
   • Bundle 2 additional products with 25% discount for 12 months
   • Free credit card annual fee waiver
   • Personal financial consultation
"""
        
        if not active:
            strategy += """
4️⃣ RE-ENGAGEMENT CAMPAIGN:
   • Welcome back bonus: $50 after first transaction
   • Personalized app notification with offers
   • Email campaign with success stories
"""
        
        strategy += """
📈 EXPECTED IMPACT:
   • 40-50% reduction in churn probability
   • Estimated value saved: ${estimated_value:,.0f}
   • ROI: $6-8 per $1 spent

📞 NEXT STEPS:
   • Contact within 24 hours
   • Document all interactions in CRM
   • Follow up after 1 week
"""
        # Calculate estimated value
        estimated_value = balance * 0.15  # 15% of balance as retention value
        strategy = strategy.replace("{estimated_value}", f"{estimated_value:.0f}")
    
    # MEDIUM RISK CUSTOMERS (Proactive Campaign)
    elif risk == "Medium Risk":
        strategy += """
🟠 PROACTIVE RETENTION CAMPAIGN
────────────────────────────────────────────────

📋 RISK ANALYSIS:
• Customer shows MODERATE risk of churn ({prob:.1%})
• {reason}
• Proactive engagement recommended within 1 week

🎯 RECOMMENDED RETENTION ACTIONS:
"""
        # Add personalized reasons
        reasons = []
        if not active:
            reasons.append("Customer is currently INACTIVE - needs re-engagement")
        else:
            reasons.append("Customer is ACTIVE but showing early warning signs")
        if products == 1:
            reasons.append("Opportunity for cross-selling")
        
        reason_text = reasons[0] if reasons else "Moderate engagement levels detected"
        strategy = strategy.replace("{reason}", reason_text)
        
        strategy += """
1️⃣ LOYALTY REWARDS:
   • Double loyalty points for 3 months
   • Birthday bonus: $10 cashback
   • Anniversary gift: Free coffee voucher

2️⃣ VALUE-ADDED SERVICES:
   • Free credit score monitoring for 6 months
   • Financial planning webinar invitation
   • Pre-approved credit line increase of $5,000

3️⃣ ENGAGEMENT CAMPAIGN:
   • Personalized email with offers
   • App push notifications
   • Monthly newsletter with tips

📈 EXPECTED IMPACT:
   • 25-35% reduction in churn probability
   • Estimated value saved: ${estimated_value:,.0f}
   • ROI: $3-5 per $1 spent

📧 NEXT STEPS:
   • Send email campaign within 1 week
   • Track open rates and engagement
   • Follow up with non-responders
"""
        estimated_value = balance * 0.10  # 10% of balance
        strategy = strategy.replace("{estimated_value}", f"{estimated_value:.0f}")
    
    # LOW RISK CUSTOMERS (Loyalty Reinforcement)
    else:
        strategy += """
🟢 LOYALTY REINFORCEMENT
────────────────────────────────────────────────

📋 CUSTOMER VALUE ANALYSIS:
• This is a LOYAL customer with LOW churn risk ({prob:.1%})
• {reason}
• Focus on maintaining satisfaction and encouraging referrals

🎯 RECOMMENDED RETENTION ACTIONS:
"""
        # Add personalized reasons
        reasons = []
        if active:
            reasons.append("Actively engaged customer")
        if products >= 3:
            reasons.append("High product adoption - loyal customer")
        if balance > 100000:
            reasons.append("High-value loyal customer")
        
        reason_text = reasons[0] if reasons else "Stable customer with good engagement"
        strategy = strategy.replace("{reason}", reason_text)
        
        strategy += """
1️⃣ APPRECIATION REWARDS:
   • $10 coffee voucher or gift card
   • Birthday bonus: $5 cashback
   • Anniversary gift: Double points for a month

2️⃣ EXCLUSIVE BENEFITS:
   • Early access to new banking features
   • Invitation to customer appreciation events
   • Priority customer support line

3️⃣ REFERRAL PROGRAM:
   • Enhanced referral bonus: $75 per successful referral
   • Referral leaderboard with quarterly prizes
   • "Friend & Family" special offers

📈 EXPECTED IMPACT:
   • 10-15% increase in loyalty metrics
   • Higher referral rate (estimated 2-3 referrals/year)
   • Maintain customer satisfaction >90%

💌 NEXT STEPS:
   • Send appreciation email this month
   • Invite to next customer event
   • Monitor for any changes in behavior
"""
    
    strategy += f"\n{'-'*60}\n"
    strategy += f"Strategy Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    strategy += f"{'='*60}"
    
    return strategy

print("✅ Template-based strategies ready")
print("   • No API keys required")
print("   • No quotas to worry about")
print("   • Works completely offline")
print("   • 100% reliable for all customers")



STEP 6: GENERATING TEMPLATE-BASED STRATEGIES
✅ Template-based strategies ready
   • No API keys required
   • No quotas to worry about
   • Works completely offline
   • 100% reliable for all customers


In [8]:
# ============================================================================
# STEP 7: GENERATE STRATEGIES FOR ALL CUSTOMERS
# ============================================================================
print("\n" + "=" * 80)
print("STEP 7: GENERATING CUSTOMER RETENTION STRATEGIES")
print("=" * 80)

strategies = []
risk_counts = {'Low Risk': 0, 'Medium Risk': 0, 'High Risk': 0}

for i, customer in enumerate(customer_profiles):
    risk_counts[customer['risk_level']] += 1
    
    # Generate strategy using templates (always works!)
    strategy_text = get_retention_strategy(customer)
    
    strategies.append({
        'customer_id': customer['customer_id'],
        'risk_level': customer['risk_level'],
        'churn_probability': f"{customer['churn_probability']:.1%}",
        'predicted_churn': customer['predicted_churn'],
        'actual_churn': customer['actual_churn'],
        'age': customer['age'],
        'products': customer['products'],
        'balance': f"${customer['balance']:,.2f}",
        'active': customer['active'],
        'credit_score': customer['credit_score'],
        'strategy': strategy_text
    })
    
    # Print progress
    if (i + 1) % 10 == 0:
        print(f"✅ Generated strategies for {i + 1}/{num_customers} customers")


STEP 7: GENERATING CUSTOMER RETENTION STRATEGIES
✅ Generated strategies for 10/50 customers
✅ Generated strategies for 20/50 customers
✅ Generated strategies for 30/50 customers
✅ Generated strategies for 40/50 customers
✅ Generated strategies for 50/50 customers


In [9]:
# ============================================================================
# STEP 8: SAVE STRATEGIES TO CSV
# ============================================================================
print("\n" + "=" * 60)
print("STEP 8: SAVING STRATEGIES")
print("=" * 60)

strategies_df = pd.DataFrame(strategies)
strategies_df.to_csv("../models/retention_strategies.csv", index=False)

print(f"✅ File saved: ../models/retention_strategies.csv")
print(f"📊 Total strategies generated: {len(strategies_df)}")
print(f"\n📋 Risk Breakdown:")
print(f"   • High Risk: {risk_counts['High Risk']} customers")
print(f"   • Medium Risk: {risk_counts['Medium Risk']} customers")
print(f"   • Low Risk: {risk_counts['Low Risk']} customers")



STEP 8: SAVING STRATEGIES
✅ File saved: ../models/retention_strategies.csv
📊 Total strategies generated: 50

📋 Risk Breakdown:
   • High Risk: 2 customers
   • Medium Risk: 15 customers
   • Low Risk: 33 customers


In [10]:
# ============================================================================
# STEP 9: DISPLAY SAMPLE STRATEGIES
# ============================================================================
print("\n" + "=" * 80)
print("STEP 9: SAMPLE STRATEGIES")
print("=" * 80)

# Show 2 sample strategies
for i in range(min(2, len(strategies))):
    print(f"\n{strategies[i]['strategy']}")



STEP 9: SAMPLE STRATEGIES


🎯 RETENTION STRATEGY FOR CUST_1001

📊 CUSTOMER SUMMARY:
────────────────────────────────────────────────
• Risk Level: Low Risk
• Churn Probability: 2.0%
• Age: 36.0 | Gender: Male | Location: Germany
• Products: 2.0 | Active Member: Yes
• Balance: $102,603.30 | Credit Score: 747.0


🟢 LOYALTY REINFORCEMENT
────────────────────────────────────────────────

📋 CUSTOMER VALUE ANALYSIS:
• This is a LOYAL customer with LOW churn risk ({prob:.1%})
• Actively engaged customer
• Focus on maintaining satisfaction and encouraging referrals

🎯 RECOMMENDED RETENTION ACTIONS:

1️⃣ APPRECIATION REWARDS:
   • $10 coffee voucher or gift card
   • Birthday bonus: $5 cashback
   • Anniversary gift: Double points for a month

2️⃣ EXCLUSIVE BENEFITS:
   • Early access to new banking features
   • Invitation to customer appreciation events
   • Priority customer support line

3️⃣ REFERRAL PROGRAM:
   • Enhanced referral bonus: $75 per successful referral
   • Referral leaderboa

In [13]:
# ============================================================================
# STEP 10: CREATE EXECUTIVE SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("📊 EXECUTIVE SUMMARY REPORT")
print("=" * 80)

total_balance_at_risk = sum(c['balance'] for c in customer_profiles if c['risk_level'] == 'High Risk')
total_high_value_at_risk = sum(1 for c in customer_profiles if c['risk_level'] == 'High Risk' and c['balance'] > 50000)

print(f"""
🎯 CAMPAIGN OVERVIEW
────────────────────────────────────────────────
Total Customers Analyzed: {num_customers}
Strategies Generated: {len(strategies_df)}
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

📊 RISK DISTRIBUTION
────────────────────────────────────────────────
🔴 High Risk:     {risk_counts['High Risk']:3d} customers ({risk_counts['High Risk']/num_customers*100:.1f}%)
🟠 Medium Risk:   {risk_counts['Medium Risk']:3d} customers ({risk_counts['Medium Risk']/num_customers*100:.1f}%)
🟢 Low Risk:      {risk_counts['Low Risk']:3d} customers ({risk_counts['Low Risk']/num_customers*100:.1f}%)

💰 FINANCIAL IMPACT
────────────────────────────────────────────────
Total Balance at Risk: ${total_balance_at_risk:,.0f}
High-Value Customers at Risk (>$50k): {total_high_value_at_risk}
Estimated Retention Value: ${total_balance_at_risk * 0.2:,.0f}

🎯 ACTION PLAN
────────────────────────────────────────────────
🔴 HIGH PRIORITY (Next 24-48 hours):
   • Contact all {risk_counts['High Risk']} high-risk customers
   • Priority calls for {total_high_value_at_risk} high-value customers
   • Offer premium retention packages

🟠 MEDIUM PRIORITY (Next 1-2 weeks):
   • Launch email campaign for {risk_counts['Medium Risk']} medium-risk customers
   • Track response rates and engagement

🟢 LOW PRIORITY (Ongoing):
   • Send appreciation rewards to {risk_counts['Low Risk']} loyal customers
   • Encourage referrals


""")

print("\n" + "=" * 80)
print("✅ PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)
print(f"\n📁 Output file: ../models/retention_strategies.csv")
print(f"📊 Total strategies: {len(strategies_df)}")
print("\n" + "=" * 80)


📊 EXECUTIVE SUMMARY REPORT

🎯 CAMPAIGN OVERVIEW
────────────────────────────────────────────────
Total Customers Analyzed: 50
Strategies Generated: 50
Date: 2026-03-10 21:01:00

📊 RISK DISTRIBUTION
────────────────────────────────────────────────
🔴 High Risk:       2 customers (4.0%)
🟠 Medium Risk:    15 customers (30.0%)
🟢 Low Risk:       33 customers (66.0%)

💰 FINANCIAL IMPACT
────────────────────────────────────────────────
Total Balance at Risk: $217,279
High-Value Customers at Risk (>$50k): 2
Estimated Retention Value: $43,456

🎯 ACTION PLAN
────────────────────────────────────────────────
🔴 HIGH PRIORITY (Next 24-48 hours):
   • Contact all 2 high-risk customers
   • Priority calls for 2 high-value customers
   • Offer premium retention packages

🟠 MEDIUM PRIORITY (Next 1-2 weeks):
   • Launch email campaign for 15 medium-risk customers
   • Track response rates and engagement

🟢 LOW PRIORITY (Ongoing):
   • Send appreciation rewards to 33 loyal customers
   • Encourage referra